In [ ]:
import logging
import os
import shutil
import tempfile

import geopandas as gpd
import pooch
import swisstopopy

# disable pooch logs
pooch.get_logger().setLevel(logging.ERROR)

In [ ]:
# spatial extent
agglom_extent_filepath = "../data/processed/zurich-extent.gpkg"

# buffer distance around the extent
buffer_dist = 2000

# to download interim data and reuse it
working_dir = "land-working-dir"

# download parameters
alti3d_res = 2

# output files
dst_buildings_gdf_filepath = "../data/processed/buildings.gpkg"
dst_dem_filepath = "../data/processed/dem.tif"
dst_tree_canopy_filepath = "../data/processed/tree-canopy.tif"

In [ ]:
region = gpd.GeoSeries(
    gpd.read_file(agglom_extent_filepath)["geometry"].buffer(buffer_dist)
)

In [ ]:
if working_dir is None:
    working_dir = tempfile.mkdtemp()
pooch_retrieve_kwargs = {"path": working_dir}

In [ ]:
bldg_gdf = swisstopopy.get_bldg_gdf(region, **pooch_retrieve_kwargs)
bldg_gdf.to_file(dst_buildings_gdf_filepath)

  0%|                                                                                                                                                           | 0/653 [00:00<?, ?it/s]/home/martibosch/data-science/uhi-cws-lausanne/zurich-heatwaves-lur/.pixi/envs/default/lib/python3.13/site-packages/rasterstats/io.py:335: NodataWarning: Setting nodata to -999; specify nodata explicitly
  warnings.warn(
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 653/653 [03:18<00:00,  3.29it/s]
/home/martibosch/libraries/swisstopopy/swisstopopy/buildings.py:182: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  bldg_height_df = pd.concat(


In [ ]:
swisstopopy.get_dem_raster(
    region,
    dst_dem_filepath,
    alti3d_res=alti3d_res,
    pooch_retrieve_kwargs=pooch_retrieve_kwargs,
)

In [ ]:
tree_canopy_tiles_dir = os.path.join(working_dir, "tree-canopy-tiles")
# classify each tile; skips tiles that already exist in dst_dir (resume-safe)
swisstopopy.get_tree_canopy_raster(
    region=region,
    dst_filepath=dst_tree_canopy_filepath,
    # dst_dir=tree_canopy_tiles_dir,
    pooch_retrieve_kwargs={"path": tree_canopy_tiles_dir},
)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 646/646 [47:57<00:00,  4.46s/it]


In [ ]:
if working_dir.startswith(tempfile.gettempdir()):
    shutil.rmtree(working_dir)